In [1]:
!pip install datasets tensorflow


In [14]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import ReduceLROnPlateau
from datasets import load_dataset
import pickle
import re


In [15]:
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
train_text = " ".join(dataset["train"]["text"])


In [16]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"=+.*?=+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

cleaned_text = clean_text(train_text)


In [17]:
VOCAB_SIZE = 12000

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts([cleaned_text])

total_words = min(VOCAB_SIZE, len(tokenizer.word_index) + 1)
print("Vocabulary Size:", total_words)


Vocabulary Size: 12000


In [18]:
SEQ_LEN = 30

token_list = tokenizer.texts_to_sequences([cleaned_text])[0]

sequences = []
for i in range(SEQ_LEN, len(token_list)):
    sequences.append(token_list[i-SEQ_LEN:i+1])

sequences = np.array(sequences)

X = sequences[:, :-1]
y = sequences[:, -1]

print("X shape:", X.shape)


X shape: (1752509, 30)


In [19]:
EMBED_DIM = 150
LSTM_UNITS = 256

model = Sequential([
    Embedding(total_words, EMBED_DIM, input_length=SEQ_LEN),
    LSTM(LSTM_UNITS, return_sequences=True),
    Dropout(0.3),
    LSTM(LSTM_UNITS),
    Dropout(0.3),
    Dense(total_words, activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    metrics=['accuracy']
)

model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [20]:
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    verbose=1
)

history = model.fit(
    X, y,
    epochs=15,
    batch_size=128,
    validation_split=0.1,
    callbacks=[lr_scheduler]
)


Epoch 1/15
12323/12323 ━━━━━━━━━━━━━━━━━━━━ 268s 22ms/step - accuracy: 0.1220 - loss: 6.5464 - val_accuracy: 0.1705 - val_loss: 6.1032 - learning_rate: 0.0010
Epoch 2/15
12323/12323 ━━━━━━━━━━━━━━━━━━━━ 272s 22ms/step - accuracy: 0.1679 - loss: 5.7826 - val_accuracy: 0.1831 - val_loss: 5.8742 - learning_rate: 0.0010
Epoch 3/15
12323/12323 ━━━━━━━━━━━━━━━━━━━━ 256s 21ms/step - accuracy: 0.1822 - loss: 5.4522 - val_accuracy: 0.1902 - val_loss: 5.7410 - learning_rate: 0.0010
Epoch 4/15
12323/12323 ━━━━━━━━━━━━━━━━━━━━ 255s 21ms/step - accuracy: 0.1925 - loss: 5.2481 - val_accuracy: 0.1945 - val_loss: 5.6761 - learning_rate: 0.0010
Epoch 5/15
12323/12323 ━━━━━━━━━━━━━━━━━━━━ 254s 21ms/step - accuracy: 0.1991 - loss: 5.1167 - val_accuracy: 0.1971 - val_loss: 5.6285 - learning_rate: 0.0010
Epoch 6/15
12323/12323 ━━━━━━━━━━━━━━━━━━━━ 254s 21ms/step - accuracy: 0.2050 - loss: 5.0138 - val_accuracy: 0.2004 - val_loss: 5.6057 - learning_rate: 0.0010
Epoch 7/15
12323/12323 ━━━━━━━━━━━━━━━━━━━━ 25

In [21]:
model.save("autocomplete_lstm_fixed.h5")

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)


In [22]:
def top_k_sampling(predictions, k=20, temperature=1.0):
    predictions = np.log(predictions + 1e-10) / temperature
    exp_preds = np.exp(predictions)
    predictions = exp_preds / np.sum(exp_preds)

    indices = np.argsort(predictions)[-k:]
    probs = predictions[indices]
    probs = probs / np.sum(probs)

    return np.random.choice(indices, p=probs)


In [23]:
def generate_text(seed_text, max_tokens=20, temperature=1.0, k=20):

    oov_index = tokenizer.word_index["<OOV>"]

    for _ in range(max_tokens):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=SEQ_LEN, padding='pre')

        predictions = model.predict(token_list, verbose=0)[0]

        # Remove OOV from prediction
        if oov_index < len(predictions):
            predictions[oov_index] = 0

        predictions = predictions / np.sum(predictions)

        predicted_index = top_k_sampling(predictions, k=k, temperature=temperature)

        output_word = tokenizer.index_word.get(predicted_index, "")

        if output_word == "":
            break

        seed_text += " " + output_word

        if output_word.endswith("."):
            break

    return seed_text


In [26]:
seed = "we have to"

print("Temp 0.7:")
print(generate_text(seed, temperature=0.7))

print("\nTemp 1.0:")
print(generate_text(seed, temperature=1.0))

print("\nTemp 1.3:")
print(generate_text(seed, temperature=1.3))


Temp 0.7:
we have to produce the call as a result of the same name the only and that i love has been released on

Temp 1.0:
we have to take some way of a low pressure of her own head and a girl for the entire american caribbean at

Temp 1.3:
we have to a small number of the cougar 's range at a rate of 4 – 3 the noisy miner is classified


In [27]:
from google.colab import files

files.download("autocomplete_lstm_fixed.h5")
files.download("tokenizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>